# NVIDIA Developer Journey — Supervised Modeling: LightGBM & XGBoost (v2)

**Goal:** Predict four 90-day developer outcomes using gradient boosting.
PCA-reduced features run in parallel as a dimensionality-reduction validation.

**Key change from v1:** Temporal leakage correction.
The original feature set included 0–30d and 30–90d recency windows that fall
*inside* the outcome window, producing near-perfect AUC that is not predictive —
it is definitional. This version constructs a leakage-safe feature set and
demonstrates the before/after AUC difference.

| Section | Contents |
|---------|----------|
| 0 | Setup & connection |
| 1 | Feature panel loading & outcome labels |
| 2 | Leakage audit — timeline and feature overlap analysis |
| 3 | Leakage-safe feature construction |
| 4 | Preprocessing & train / val / test split |
| 5 | PCA analysis (variance scree, component selection) |
| 6 | Leakage demonstration (leaky vs. safe AUC comparison) |
| 7 | LightGBM — leakage-safe features |
| 8 | XGBoost — leakage-safe features |
| 9 | LightGBM & XGBoost — PCA features (validation) |
| 10 | Model comparison table (AUC, PR-AUC, Brier) |
| 11 | Feature importance (gain) |
| 12 | SHAP analysis |
| 13 | Calibration (reliability diagrams + Platt scaling) |
| 14 | Score output (predicted probabilities per developer) |

**Outcome labels** (90-day horizon, cutoff = anchor_date − 90 days):
- `retained_90d` — any activity in [cutoff, anchor]
- `deepened_90d` — Build/Champion signal or ≥ 20 % score growth post-cutoff
- `expanded_90d` — touched a new persona lane post-cutoff
- `churned_90d` — active pre-cutoff, zero activity post-cutoff

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

DB_PATH        = "developer_project.duckdb"
SAMPLE_PARQUET = "sample_dev_profile_final_v4.parquet"
RANDOM_STATE   = 42
np.random.seed(RANDOM_STATE)

if Path(DB_PATH).exists():
    con = duckdb.connect(DB_PATH, read_only=True)
    DATA_SOURCE = "duckdb"
    print(f"Connected to {DB_PATH}")
elif Path(SAMPLE_PARQUET).exists():
    con = duckdb.connect(":memory:")
    con.execute(
        "CREATE TABLE dev_profile_final_v4 AS SELECT * FROM read_parquet('"
        + SAMPLE_PARQUET + "')"
    )
    DATA_SOURCE = "parquet"
    print(f"Loaded sample parquet: {SAMPLE_PARQUET}")
else:
    raise FileNotFoundError(
        "Neither " + DB_PATH + " nor " + SAMPLE_PARQUET + " found. "
        "Run FeatureEngineering_v3.ipynb first."
    )

## 1. Feature panel loading & outcome labels

In [ ]:
df_raw = con.execute("SELECT * FROM dev_profile_final_v4").df()
print(f"dev_profile_final_v4: {len(df_raw):,} rows  |  {df_raw.shape[1]} columns")

# Ordinal-encode categorical columns needed later
USER_TYPE_MAP   = {"tourist": 0, "free_email_user": 1, "real_user": 2}
STAGE_MAP       = {"None": 0, "Discover": 1, "Learn": 2, "Evaluate": 3, "Build": 4, "Champion": 5}
PERSONA_ORD_MAP = {"Unknown": 0, "CUDA": 1, "GenAI": 2, "Robotics": 3,
                   "Simulation": 4, "Learning/Community": 5}

for col, mapping in [("user_type", USER_TYPE_MAP),
                     ("max_stage_reached", STAGE_MAP),
                     ("persona", PERSONA_ORD_MAP)]:
    if col in df_raw.columns:
        df_raw[col + "_ord"] = df_raw[col].map(mapping).fillna(0).astype(int)

print("Ordinal-encoded: user_type_ord, max_stage_reached_ord, persona_ord")

In [ ]:
# ── Outcome labels (90-day cutoff) ─────────────────────────────────────────
TARGETS = ["retained_90d", "deepened_90d", "expanded_90d", "churned_90d"]

if DATA_SOURCE == "duckdb":
    df_labels = con.execute("""
    WITH max_dt AS (SELECT MAX(activity_date) AS anchor_date FROM activity_labeled_v2),
    cutoff AS (
        SELECT anchor_date - INTERVAL 90 DAY AS cutoff_date, anchor_date FROM max_dt
    ),
    pre AS (
        SELECT
            a.developer_id,
            COUNT(*)                                                                  AS pre_count,
            SUM(a.activity_score)                                                     AS pre_score,
            MAX(CASE WHEN a.journey_signal IN ('Build','Champion') THEN 1 ELSE 0 END) AS pre_had_build,
            COUNT(DISTINCT
                CASE WHEN a.cuda_persona_score               > 0 THEN 'cuda'
                     WHEN a.genai_persona_score              > 0 THEN 'genai'
                     WHEN a.robotics_persona_score           > 0 THEN 'robotics'
                     WHEN a.simulation_persona_score         > 0 THEN 'simulation'
                     WHEN a.learning_community_persona_score > 0 THEN 'learning'
                END
            )                                                                         AS pre_lane_count
        FROM activity_labeled_v2 a, cutoff
        WHERE a.activity_date <= cutoff.cutoff_date
        GROUP BY a.developer_id
    ),
    post AS (
        SELECT
            a.developer_id,
            COUNT(*)                                                                  AS post_count,
            SUM(a.activity_score)                                                     AS post_score,
            MAX(CASE WHEN a.journey_signal IN ('Build','Champion') THEN 1 ELSE 0 END) AS post_had_build,
            COUNT(DISTINCT
                CASE WHEN a.cuda_persona_score               > 0 THEN 'cuda'
                     WHEN a.genai_persona_score              > 0 THEN 'genai'
                     WHEN a.robotics_persona_score           > 0 THEN 'robotics'
                     WHEN a.simulation_persona_score         > 0 THEN 'simulation'
                     WHEN a.learning_community_persona_score > 0 THEN 'learning'
                END
            )                                                                         AS post_lane_count
        FROM activity_labeled_v2 a, cutoff
        WHERE a.activity_date > cutoff.cutoff_date
        GROUP BY a.developer_id
    )
    SELECT
        u.developer_id,
        CASE WHEN COALESCE(post.post_count, 0) > 0 THEN 1 ELSE 0 END                AS retained_90d,
        CASE
            WHEN COALESCE(post.post_had_build, 0) = 1                        THEN 1
            WHEN COALESCE(post.post_score, 0) > COALESCE(pre.pre_score, 0) * 1.20
             AND COALESCE(post.post_count, 0) > 0                            THEN 1
            ELSE 0
        END                                                                           AS deepened_90d,
        CASE
            WHEN COALESCE(post.post_lane_count, 0) > COALESCE(pre.pre_lane_count, 0)
             AND COALESCE(post.post_count, 0) > 0                            THEN 1
            ELSE 0
        END                                                                           AS expanded_90d,
        CASE
            WHEN COALESCE(pre.pre_count,  0) > 0
             AND COALESCE(post.post_count, 0) = 0                            THEN 1
            ELSE 0
        END                                                                           AS churned_90d
    FROM developer_universe_v2 u
    LEFT JOIN pre  USING (developer_id)
    LEFT JOIN post USING (developer_id)
    """).df()
else:
    # Parquet fallback — approximate from available columns
    df_labels = pd.DataFrame({
        "developer_id": df_raw["developer_id"],
        "retained_90d": (df_raw.get("activity_count_90_180d",
                          pd.Series(0, index=df_raw.index)) > 0).astype(int),
        "deepened_90d": (df_raw.get("build_count_90_180d",
                          pd.Series(0, index=df_raw.index)) > 0).astype(int),
        "expanded_90d": (df_raw.get("mixed_persona_flag",
                          pd.Series(0, index=df_raw.index)) == 1).astype(int),
        "churned_90d":  (
            (df_raw.get("activity_count_90_180d", pd.Series(0, index=df_raw.index)) > 0) &
            (df_raw.get("has_activity_90_180d",   pd.Series(0, index=df_raw.index)) == 0)
        ).astype(int),
    })

df_base = df_raw.merge(df_labels, on="developer_id", how="left")
print("Label distribution (positive rates):")
display(df_base[TARGETS].mean().rename("rate").to_frame().T)

## 2. Leakage audit — timeline and feature overlap analysis

### The problem

The outcome labels are defined on a **90-day outcome window**:
`[anchor_date − 90 days, anchor_date]`.

Several feature columns in `dev_profile_final_v4` describe activity from
**the same window** or are derived from state computed at `anchor_date`:

| Feature group | Window covered | Overlap with outcome? |
|---|---|---|
| `*_0_30d` | anchor − 30d → anchor | **YES — fully inside** |
| `*_30_90d` | anchor − 90d → anchor − 30d | **YES — fully inside** |
| `*_90_180d` | anchor − 180d → anchor − 90d | No — ends at cutoff |
| `lifetime_*` | all time → anchor | Partially (last 90d included) |
| `dormant_flag`, `days_since_last_activity` | computed at anchor | **YES — encodes outcome** |
| `recent_build_flag`, `recent_champion_flag` | based on 0–30d | **YES** |
| `weighted_recent_activity` | 0.6×(0–30d) + 0.3×(30–90d) + 0.1×(90–180d) | **YES** |
| Persona shares, `user_type`, `max_stage_reached` | lifetime classification | Minimal |

A model that sees `activity_count_0_30d` can trivially learn:
`retained_90d = 1` if `activity_count_0_30d > 0` — the label **is** the feature.

In [ ]:
# ── Timeline diagram ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))
ax.set_xlim(-200, 20)
ax.set_ylim(-0.5, 5.5)
ax.axis("off")
ax.set_title("Feature window vs. outcome window overlap", fontsize=12, pad=10)

# Reference line
ax.axvline(0,   color="black", lw=1.5, linestyle="--")
ax.axvline(-90, color="red",   lw=1.5, linestyle="--")

ax.text(2, 5.2, "anchor_date", fontsize=8, color="black", ha="left")
ax.text(-95, 5.2, "cutoff\n(anchor-90d)", fontsize=8, color="red", ha="right")

bars = [
    (-90, 90, 0.6, "#f44336", "Outcome window [−90d, 0] ← LABEL DEFINED HERE"),
    (-30, 30, 1.4, "#f44336", "0–30d features (LEAKY — fully inside outcome)"),
    (-90, 60, 2.2, "#ff7043", "30–90d features (LEAKY — fully inside outcome)"),
    (-180, 90, 3.0, "#4caf50", "90–180d features (SAFE — ends at cutoff)"),
    (-180, 180, 3.8, "#9e9e9e", "Lifetime features (mild leakage — includes last 90d)"),
]

for x_start, width, y, color, label in bars:
    ax.barh(y, width, left=x_start, height=0.55, color=color, alpha=0.75, edgecolor="white")
    ax.text(x_start - 2, y, label, va="center", ha="right", fontsize=7.5, color="black")

ax.set_xlabel("Days relative to anchor_date")
plt.tight_layout()
plt.show()

# ── Smoking-gun correlation: activity_count_0_30d vs. retained_90d ──────────
if "activity_count_0_30d" in df_base.columns and "retained_90d" in df_base.columns:
    corr_30 = df_base["activity_count_0_30d"].clip(upper=1).corr(df_base["retained_90d"])
    corr_90 = df_base.get("activity_count_30_90d",
                           pd.Series(0, index=df_base.index)).clip(upper=1).corr(
                               df_base["retained_90d"])
    corr_safe = df_base.get("activity_count_90_180d",
                              pd.Series(0, index=df_base.index)).clip(upper=1).corr(
                                  df_base["retained_90d"])
    print("Point-biserial correlation with retained_90d:")
    print(f"  has_activity_0_30d  (LEAKY):  {corr_30:.4f}  ← near-perfect by construction")
    print(f"  has_activity_30_90d (LEAKY):  {corr_90:.4f}")
    print(f"  has_activity_90_180d (SAFE):  {corr_safe:.4f}  ← genuine predictive signal")

## 3. Leakage-safe feature construction

**Rules applied:**
1. Drop all `*_0_30d` and `*_30_90d` columns.
2. Drop recency-encoded lifecycle flags computed at `anchor_date`:
   `dormant_flag`, `at_risk_flag`, `cooling_flag`, `days_since_last_activity`,
   `days_since_last_meaningful_week`, `is_activated`,
   `recent_build_flag`, `recent_champion_flag`,
   `weighted_recent_*`, `*_velocity_*`, `effort_recency_weight`,
   `developer_effort_rank`, `behavior_journey_rank_30d`, `current_journey_rank_30d`.
3. Derive **pre-cutoff proxies** for high-value lifetime counts where 0–30d and
   30–90d breakdowns are available:
   `pre_activity_count = lifetime_count − 0_30d − 30_90d`
4. Keep `*_90_180d` window features, persona shares, `user_type_ord`,
   `max_stage_reached_ord`, and lifetime specialty counts unlikely to be
   dominated by the last 90 days (DLI, forum, hackathon, bug, webinar).

In [ ]:
df = df_base.copy()

# ── Step 1: derive pre-cutoff proxies ──────────────────────────────────────
# For each count feature where we have both lifetime and 0_30d / 30_90d splits:
PROXY_PAIRS = [
    ("lifetime_activity_count",                "activity_count_0_30d",    "activity_count_30_90d"),
    ("lifetime_build_count",                   "build_count_0_30d",       "build_count_30_90d"),
    ("lifetime_high_effort_count",             "high_effort_count_0_30d", "high_effort_count_30_90d"),
    ("lifetime_total_confidence_weighted_effort",
     "total_confidence_weighted_effort_0_30d", "total_confidence_weighted_effort_30_90d"),
]

for lifetime_col, w0_col, w30_col in PROXY_PAIRS:
    if lifetime_col in df.columns:
        w0  = df[w0_col].fillna(0)  if w0_col  in df.columns else 0
        w30 = df[w30_col].fillna(0) if w30_col in df.columns else 0
        proxy_name = "pre_cutoff_" + lifetime_col.replace("lifetime_", "")
        df[proxy_name] = (df[lifetime_col].fillna(0) - w0 - w30).clip(lower=0)
        df["log_" + proxy_name] = np.log1p(df[proxy_name])

# Derived share / ratio features (safe, computed from pre-cutoff proxies)
if "pre_cutoff_activity_count" in df.columns:
    pc_act = df["pre_cutoff_activity_count"].replace(0, np.nan)
    df["pre_cutoff_build_share"]       = (df.get("pre_cutoff_build_count", 0) / pc_act).fillna(0)
    df["pre_cutoff_high_effort_share"] = (df.get("pre_cutoff_high_effort_count", 0) / pc_act).fillna(0)

print("Pre-cutoff proxy columns created:")
proxy_cols = [c for c in df.columns if c.startswith("pre_cutoff_") or c.startswith("log_pre_cutoff_")]
print("  " + ", ".join(proxy_cols))

# ── Step 2: define the safe feature list ───────────────────────────────────
# 90_180d window features (end at cutoff boundary — safe)
SAFE_90_180D = [c for c in df.columns if c.endswith("_90_180d")]

# Lifetime specialty counts unlikely to be dominated by last 90 days
SAFE_LIFETIME_SPECIALTY = [
    "lifetime_dli_training_count", "lifetime_webinar_count", "lifetime_forum_count",
    "lifetime_bug_count", "lifetime_hackathon_count", "lifetime_api_count",
    "lifetime_devzone_download_count", "lifetime_ngc_download_count",
    "lifetime_activity_score_avg", "lifetime_avg_effort_rank", "lifetime_max_effort_rank",
    "lifetime_avg_score_effort_gap", "lifetime_score_effort_misalignment_count",
    "lifetime_unique_activity_types", "lifetime_unique_modalities",
    "lifetime_discover_count", "lifetime_learn_count", "lifetime_evaluate_count",
    "lifetime_champion_count",
    "build_share_lifetime", "high_effort_share_lifetime",
    "effort_per_activity_lifetime", "score_effort_misalignment_share_lifetime",
    "log_lifetime_activity_score_sum",
]

# Persona shares (derived from lifetime, stable across the 90-day window)
SAFE_PERSONA = [
    "cuda_share", "genai_share", "robotics_share",
    "simulation_share", "learning_community_share",
    "persona_entropy", "mixed_persona_flag", "persona_confidence",
]

# Ordinal-encoded stable classifications
SAFE_CATEGORICAL = [
    c for c in ["user_type_ord", "max_stage_reached_ord", "persona_ord"]
    if c in df.columns
]

# Pre-cutoff proxies
SAFE_PROXIES = [c for c in df.columns
                if c.startswith("pre_cutoff_") or c.startswith("log_pre_cutoff_")]

SAFE_FEATURES = sorted(set(
    SAFE_90_180D + SAFE_LIFETIME_SPECIALTY + SAFE_PERSONA +
    SAFE_CATEGORICAL + SAFE_PROXIES
))

# Keep only columns that exist and have some variance
SAFE_FEATURES = [f for f in SAFE_FEATURES if f in df.columns
                 and df[f].nunique() > 1]

print(f"\nLeakage-safe feature set: {len(SAFE_FEATURES)} features")
print(f"  90_180d window:        {len(SAFE_90_180D)} cols")
print(f"  Lifetime specialty:    {len(SAFE_LIFETIME_SPECIALTY)} cols")
print(f"  Persona shares:        {len(SAFE_PERSONA)} cols")
print(f"  Ordinal categoricals:  {len(SAFE_CATEGORICAL)} cols")
print(f"  Pre-cutoff proxies:    {len(SAFE_PROXIES)} cols")

## 4. Preprocessing & train / val / test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

X_raw = df[SAFE_FEATURES].copy()
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)

# Clip at 99th percentile per column
for col in X_raw.columns:
    p99 = X_raw[col].quantile(0.99)
    if pd.notna(p99):
        X_raw[col] = X_raw[col].clip(upper=p99)

imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()
X_imp    = imputer.fit_transform(X_raw)
X_scaled = scaler.fit_transform(X_imp)

X_safe = pd.DataFrame(X_scaled, columns=SAFE_FEATURES, index=df.index)
y      = df[TARGETS].fillna(0).astype(int)

# 70 / 15 / 15 stratified on retained_90d
X_tr_val, X_test,  y_tr_val, y_test  = train_test_split(
    X_safe, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y["retained_90d"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tr_val, y_tr_val, test_size=0.15 / 0.85,
    random_state=RANDOM_STATE, stratify=y_tr_val["retained_90d"]
)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")
print(f"Test positive rates:")
display(y_test.mean().rename("rate").to_frame().T)

## 5. PCA analysis — variance scree & component selection

PCA runs on the **leakage-safe** feature set.
If PCA-reduced models achieve similar AUC to full safe-feature models,
the safe behavioral signal is highly compressible.

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = int(np.searchsorted(cumvar, 0.90)) + 1
n_95 = int(np.searchsorted(cumvar, 0.95)) + 1
n_99 = int(np.searchsorted(cumvar, 0.99)) + 1
print(f"Safe features: {len(SAFE_FEATURES)}")
print(f"  Components for 90% variance: {n_90}")
print(f"  Components for 95% variance: {n_95}")
print(f"  Components for 99% variance: {n_99}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(range(1, min(len(cumvar) + 1, 51)),
             pca_full.explained_variance_ratio_[:50] * 100,
             marker="o", markersize=3, linewidth=1.2)
axes[0].set_title("Per-component explained variance (first 50)")
axes[0].set_xlabel("Component"); axes[0].set_ylabel("Variance explained (%)")

axes[1].plot(range(1, len(cumvar) + 1), cumvar * 100, linewidth=1.5)
for thresh, n_comp, col in [(90, n_90, "green"), (95, n_95, "orange"), (99, n_99, "red")]:
    axes[1].axhline(thresh, color=col, ls="--", alpha=0.7, label=f"{thresh}% @ {n_comp} PCs")
    axes[1].axvline(n_comp, color=col, ls=":",  alpha=0.5)
axes[1].set_title("Cumulative explained variance (safe features)")
axes[1].set_xlabel("Number of components"); axes[1].set_ylabel("Cumulative variance (%)")
axes[1].legend()
plt.tight_layout()
plt.show()

N_PCA = n_95
print(f"\nUsing {N_PCA} PCA components (95% variance threshold)")

In [ ]:
pca = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)
print(f"PCA train shape: {X_train_pca.shape}")

## 6. Leakage demonstration — leaky vs. safe AUC

A quick LightGBM fit on the **leaky** feature set (all features including 0–30d / 30–90d windows)
is compared against the same model trained on the **safe** feature set.

The gap confirms that the original AUC ≈ 1.0 was definitional, not predictive.

In [ ]:
try:
    import lightgbm as lgb
except ImportError:
    raise ImportError("pip install lightgbm")

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def pos_weight(y_ser):
    n_neg = (y_ser == 0).sum()
    n_pos = (y_ser == 1).sum()
    return n_neg / max(n_pos, 1)

def eval_binary(model, X, y_true, label=""):
    prob = model.predict_proba(X)[:, 1]
    auc   = roc_auc_score(y_true, prob)
    ap    = average_precision_score(y_true, prob)
    brier = brier_score_loss(y_true, prob)
    if label:
        print(f"  {label:<20} AUC={auc:.4f}  PR-AUC={ap:.4f}  Brier={brier:.4f}")
    return {"auc": auc, "ap": ap, "brier": brier, "prob": prob}

QUICK_LGB = dict(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    num_leaves=31, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)

# ── Build leaky feature matrix (all features including 0_30d, 30_90d, dormancy flags) ──
ALL_LEAKY_CANDIDATES = [
    "activity_count_0_30d", "activity_count_30_90d",
    "has_activity_0_30d", "has_activity_30_90d",
    "build_count_0_30d", "build_count_30_90d",
    "high_effort_count_0_30d", "high_effort_count_30_90d",
    "days_since_last_activity", "dormant_flag", "at_risk_flag", "cooling_flag",
    "recent_build_flag", "recent_champion_flag",
    "weighted_recent_activity", "weighted_recent_build",
    "activity_velocity_0_30_vs_30_90",
]
leaky_cols = [c for c in ALL_LEAKY_CANDIDATES if c in df.columns]
all_cols   = leaky_cols + SAFE_FEATURES
all_cols   = list(dict.fromkeys(all_cols))  # deduplicate, preserve order

X_leaky_raw = df[all_cols].copy().replace([np.inf, -np.inf], np.nan)
for col in X_leaky_raw.columns:
    p99 = X_leaky_raw[col].quantile(0.99)
    if pd.notna(p99):
        X_leaky_raw[col] = X_leaky_raw[col].clip(upper=p99)

imp_l  = SimpleImputer(strategy="median")
X_leaky_all = pd.DataFrame(imp_l.fit_transform(X_leaky_raw),
                            columns=all_cols, index=df.index)

X_l_train = X_leaky_all.iloc[X_train.index]
X_l_val   = X_leaky_all.iloc[X_val.index]
X_l_test  = X_leaky_all.iloc[X_test.index]

print("=" * 60)
print("LEAKAGE DEMONSTRATION — retained_90d")
print("=" * 60)
print("\n[Leaky features — includes 0_30d, 30_90d, dormancy flags]")
m_leaky = lgb.LGBMClassifier(scale_pos_weight=pos_weight(y_train["retained_90d"]),
                               **QUICK_LGB)
m_leaky.fit(X_l_train, y_train["retained_90d"],
            eval_set=[(X_l_val, y_val["retained_90d"])],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
eval_binary(m_leaky, X_l_test, y_test["retained_90d"],
            label="retained_90d test")

print("\n[Safe features only — no overlap with outcome window]")
m_safe_demo = lgb.LGBMClassifier(scale_pos_weight=pos_weight(y_train["retained_90d"]),
                                  **QUICK_LGB)
m_safe_demo.fit(X_train, y_train["retained_90d"],
                eval_set=[(X_val, y_val["retained_90d"])],
                callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
eval_binary(m_safe_demo, X_test, y_test["retained_90d"],
            label="retained_90d test")

print("\nThe AUC gap between leaky and safe is the leakage contribution.")
print("Safe AUC reflects genuine predictive signal from pre-cutoff behavior.")

## 7. LightGBM — leakage-safe features

Four binary classifiers, one per outcome. `scale_pos_weight` handles class imbalance.
All features are strictly pre-cutoff (no overlap with the 90-day outcome window).

In [ ]:
LGB_PARAMS = dict(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    num_leaves        = 63,
    min_child_samples = 50,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbosity         = -1,
)

lgb_models  = {}
lgb_results = {}

print("LightGBM — leakage-safe features")
print("-" * 58)
for tgt in TARGETS:
    spw = pos_weight(y_train[tgt])
    m   = lgb.LGBMClassifier(scale_pos_weight=spw, **LGB_PARAMS)
    m.fit(
        X_train, y_train[tgt],
        eval_set=[(X_val, y_val[tgt])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    lgb_models[tgt]  = m
    lgb_results[tgt] = {
        "val":  eval_binary(m, X_val,  y_val[tgt],  f"{tgt} val"),
        "test": eval_binary(m, X_test, y_test[tgt], f"{tgt} test"),
    }

print("\nBest iteration per target:")
for tgt, m in lgb_models.items():
    print(f"  {tgt}: {m.best_iteration_} trees")

## 8. XGBoost — leakage-safe features

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    raise ImportError("pip install xgboost")

XGB_PARAMS = dict(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 6,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    eval_metric           = "auc",
    early_stopping_rounds = 50,
    random_state          = RANDOM_STATE,
    n_jobs                = -1,
    verbosity             = 0,
)

xgb_models  = {}
xgb_results = {}

print("XGBoost — leakage-safe features")
print("-" * 58)
for tgt in TARGETS:
    spw = pos_weight(y_train[tgt])
    m   = xgb.XGBClassifier(scale_pos_weight=spw, **XGB_PARAMS)
    m.fit(
        X_train, y_train[tgt],
        eval_set=[(X_val, y_val[tgt])],
        verbose=False,
    )
    xgb_models[tgt]  = m
    xgb_results[tgt] = {
        "val":  eval_binary(m, X_val,  y_val[tgt],  f"{tgt} val"),
        "test": eval_binary(m, X_test, y_test[tgt], f"{tgt} test"),
    }

print("\nBest iteration per target:")
for tgt, m in xgb_models.items():
    print(f"  {tgt}: {m.best_iteration} trees")

## 9. LightGBM & XGBoost — PCA features (validation)

Running both models on the `N_PCA`-component PCA of the safe feature set.
A small AUC gap (< 0.02) means the safe behavioral signal is highly compressible
and a simpler PCA-based production pipeline could be viable.

In [ ]:
lgb_pca_results = {}
xgb_pca_results = {}

print(f"LightGBM — PCA ({N_PCA} components, safe features)")
print("-" * 58)
for tgt in TARGETS:
    spw = pos_weight(y_train[tgt])
    m   = lgb.LGBMClassifier(scale_pos_weight=spw, **LGB_PARAMS)
    m.fit(
        X_train_pca, y_train[tgt],
        eval_set=[(X_val_pca, y_val[tgt])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    lgb_pca_results[tgt] = {
        "val":  eval_binary(m, X_val_pca,  y_val[tgt],  f"{tgt} val"),
        "test": eval_binary(m, X_test_pca, y_test[tgt], f"{tgt} test"),
    }

print(f"\nXGBoost — PCA ({N_PCA} components, safe features)")
print("-" * 58)
for tgt in TARGETS:
    spw = pos_weight(y_train[tgt])
    m   = xgb.XGBClassifier(scale_pos_weight=spw, **XGB_PARAMS)
    m.fit(
        X_train_pca, y_train[tgt],
        eval_set=[(X_val_pca, y_val[tgt])],
        verbose=False,
    )
    xgb_pca_results[tgt] = {
        "val":  eval_binary(m, X_val_pca,  y_val[tgt],  f"{tgt} val"),
        "test": eval_binary(m, X_test_pca, y_test[tgt], f"{tgt} test"),
    }

## 10. Model comparison table (test-set metrics)

All metrics are on the **leakage-safe** feature sets.
`Δ AUC (full − PCA)` shows the marginal value of the full safe feature space over PCA.

In [ ]:
rows = []
for tgt in TARGETS:
    lf = lgb_results[tgt]["test"]
    xf = xgb_results[tgt]["test"]
    lp = lgb_pca_results[tgt]["test"]
    xp = xgb_pca_results[tgt]["test"]
    rows.append({
        "target":          tgt,
        "LGB_safe_AUC":    lf["auc"],
        "XGB_safe_AUC":    xf["auc"],
        "LGB_PCA_AUC":     lp["auc"],
        "XGB_PCA_AUC":     xp["auc"],
        "LGB_safe_PRAUC":  lf["ap"],
        "XGB_safe_PRAUC":  xf["ap"],
        "LGB_safe_Brier":  lf["brier"],
        "XGB_safe_Brier":  xf["brier"],
        "Delta_AUC_LGB":   lf["auc"] - lp["auc"],
        "Delta_AUC_XGB":   xf["auc"] - xp["auc"],
    })

cmp_df = pd.DataFrame(rows).set_index("target")

def _highlight(v):
    if isinstance(v, float):
        if v > 0.02:  return "color: green"
        if v < -0.01: return "color: red"
    return ""

display(
    cmp_df.style.format("{:.4f}")
    .applymap(_highlight, subset=["Delta_AUC_LGB", "Delta_AUC_XGB"])
    .set_caption("Test-set metrics — leakage-safe features (LightGBM & XGBoost vs PCA)")
)
print("\nΔ AUC > 0.02 → full safe features add meaningful signal over PCA")
print("Δ AUC < 0.01 → PCA is a viable compact alternative")

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4))
labels = ["LGB\nSafe", "XGB\nSafe", "LGB\nPCA", "XGB\nPCA"]
colors = ["#1565C0", "#0D47A1", "#90CAF9", "#64B5F6"]

for ax, tgt in zip(axes, TARGETS):
    aucs = [
        lgb_results[tgt]["test"]["auc"],
        xgb_results[tgt]["test"]["auc"],
        lgb_pca_results[tgt]["test"]["auc"],
        xgb_pca_results[tgt]["test"]["auc"],
    ]
    bars = ax.bar(labels, aucs, color=colors, width=0.6, edgecolor="white")
    ax.set_ylim(max(0, min(aucs) - 0.05), min(1.0, max(aucs) + 0.1))
    ax.set_title(tgt.replace("_", " "), fontsize=10)
    ax.set_ylabel("ROC-AUC" if ax == axes[0] else "")
    ax.axhline(0.5, color="gray", ls="--", alpha=0.5)
    for bar, auc in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{auc:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("ROC-AUC — leakage-safe feature sets (test set)", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 11. Feature importance (gain)

Gain-based importance averaged across the four LightGBM models and four XGBoost models.
These importances are interpretable because the feature set is leakage-free:
a high-importance feature genuinely predicts future outcomes from past behavior.

In [ ]:
def mean_importance(models, feat_names, top_n=25):
    arr = np.zeros(len(feat_names))
    for m in models.values():
        arr += m.feature_importances_
    s = pd.Series(arr / len(models), index=feat_names).sort_values(ascending=False)
    return s.head(top_n)

lgb_imp = mean_importance(lgb_models, SAFE_FEATURES)
xgb_imp = mean_importance(xgb_models, SAFE_FEATURES)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, imp, title in [
    (axes[0], lgb_imp, "LightGBM — Top 25 features (avg gain, safe set)"),
    (axes[1], xgb_imp, "XGBoost  — Top 25 features (avg gain, safe set)"),
]:
    ax.barh(imp.index[::-1], imp.values[::-1], color="#1565C0", edgecolor="white")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Mean importance (gain)")
    ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Per-target top-10 importance (LightGBM, safe features)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, tgt in zip(axes.flat, TARGETS):
    m   = lgb_models[tgt]
    imp = pd.Series(m.feature_importances_,
                    index=SAFE_FEATURES).sort_values(ascending=False).head(10)
    ax.barh(imp.index[::-1], imp.values[::-1], color="#42A5F5", edgecolor="white")
    ax.set_title(f"LightGBM importance: {tgt}", fontsize=10)
    ax.set_xlabel("Gain"); ax.tick_params(axis="y", labelsize=8)
plt.suptitle("Top-10 features per outcome — LightGBM safe features", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## 12. SHAP analysis

SHAP values show *which specific feature values* drove each prediction.
Because the feature set is leakage-free, high-SHAP features reflect genuine
pre-cutoff behavioral signals that cause future outcomes.

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not installed — skipping. Install with: pip install shap")

if HAS_SHAP:
    shap_n = min(5000, len(X_test))
    rng    = np.random.default_rng(RANDOM_STATE)
    s_idx  = rng.choice(len(X_test), shap_n, replace=False)
    X_shap = X_test.iloc[s_idx]

    print(f"Computing SHAP on {shap_n:,} test samples (leakage-safe LightGBM) ...")
    for tgt in TARGETS:
        explainer = shap.TreeExplainer(lgb_models[tgt])
        sv        = explainer.shap_values(X_shap)
        sv_pos    = sv[1] if isinstance(sv, list) else sv
        print(f"\n--- SHAP summary: {tgt} ---")
        shap.summary_plot(sv_pos, X_shap, feature_names=SAFE_FEATURES,
                          max_display=20, show=True, plot_size=(10, 6))

## 13. Calibration — reliability diagrams + Platt scaling

Well-calibrated probabilities are required for operational use (targeting thresholds,
expected-value scoring). We plot reliability diagrams, then apply Platt (sigmoid)
calibration to any model where the curve deviates significantly from the diagonal.

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# ── Reliability diagrams ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, tgt in zip(axes.flat, TARGETS):
    y_true = y_test[tgt].values
    if len(np.unique(y_true)) < 2:
        ax.set_title(f"{tgt} — no positive cases"); continue

    for label, res in [
        ("LGB safe",  lgb_results[tgt]["test"]),
        ("XGB safe",  xgb_results[tgt]["test"]),
        ("LGB PCA",   lgb_pca_results[tgt]["test"]),
    ]:
        fp, mp = calibration_curve(y_true, res["prob"], n_bins=10)
        ax.plot(mp, fp, marker="o", markersize=4, label=label)

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect")
    ax.set_title(f"Reliability diagram: {tgt}", fontsize=10)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend(fontsize=8)

plt.suptitle("Calibration — reliability diagrams (test set, safe features)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Platt scaling: apply to every LightGBM model, re-evaluate ──────────────
calibrated_models = {}
print("Platt-scaled LightGBM (sigmoid calibration on val set):")
print("-" * 58)
for tgt in TARGETS:
    cal = CalibratedClassifierCV(lgb_models[tgt], method="sigmoid", cv="prefit")
    cal.fit(X_val, y_val[tgt])
    calibrated_models[tgt] = cal
    prob_cal  = cal.predict_proba(X_test)[:, 1]
    brier_cal = brier_score_loss(y_test[tgt], prob_cal)
    brier_raw = lgb_results[tgt]["test"]["brier"]
    auc_cal   = roc_auc_score(y_test[tgt], prob_cal)
    delta_b   = brier_cal - brier_raw
    print(f"  {tgt:<20} AUC={auc_cal:.4f}  "
          f"Brier raw={brier_raw:.4f} → calibrated={brier_cal:.4f}  "
          f"(Δ={delta_b:+.4f})")

## 14. Score output — predicted probabilities per developer

Uses the **calibrated LightGBM** models (safe features) as the primary score.
Raw LightGBM and XGBoost scores are also included for comparison.

These scores are operationally safe to use for targeting because:
- Features are strictly pre-cutoff (no leakage)
- Probabilities are Platt-calibrated (match observed frequencies)

In [ ]:
X_all = X_safe  # full scaled safe feature matrix

score_dict = {}
for tgt in TARGETS:
    score_dict[f"lgb_cal_p_{tgt}"]  = calibrated_models[tgt].predict_proba(X_all)[:, 1]
    score_dict[f"lgb_raw_p_{tgt}"]  = lgb_models[tgt].predict_proba(X_all)[:, 1]
    score_dict[f"xgb_raw_p_{tgt}"]  = xgb_models[tgt].predict_proba(X_all)[:, 1]

df_scores = pd.DataFrame(score_dict, index=df.index)
df_scores.insert(0, "developer_id", df["developer_id"].values)

# Top predicted outcome per developer (calibrated LightGBM)
cal_cols = [f"lgb_cal_p_{t}" for t in TARGETS]
df_scores["top_outcome"] = (
    df_scores[cal_cols]
    .idxmax(axis=1)
    .str.replace("lgb_cal_p_", "", regex=False)
)

print(f"Score table: {df_scores.shape[0]:,} developers  |  {df_scores.shape[1]} columns")
display(df_scores[[f"lgb_cal_p_{t}" for t in TARGETS]].describe().T)

out_path = "developer_scores.parquet"
df_scores.to_parquet(out_path, index=False)
print(f"\nScores saved to: {out_path}")

In [ ]:
# Score distribution: calibrated LGB positive vs. negative class
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, tgt in zip(axes.flat, TARGETS):
    col     = f"lgb_cal_p_{tgt}"
    y_true  = df_scores["developer_id"].map(
        df_base.set_index("developer_id")[tgt]
    ).fillna(0).astype(int)
    pos     = df_scores.loc[y_true == 1, col]
    neg     = df_scores.loc[y_true == 0, col]
    ax.hist(neg.values, bins=50, alpha=0.5, color="#90CAF9", density=True, label="Negative")
    ax.hist(pos.values, bins=50, alpha=0.6, color="#1565C0", density=True, label="Positive")
    ax.set_title(f"Score distribution: {tgt}", fontsize=10)
    ax.set_xlabel("Calibrated probability"); ax.set_ylabel("Density")
    ax.legend(fontsize=9)
plt.suptitle("LightGBM calibrated score distributions (all developers)", y=1.02)
plt.tight_layout()
plt.show()

con.close()
print("Done. Connection closed.")